In [3]:

import pandas as pd
from scipy.stats import randint

from sklearn.datasets import fetch_openml
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [4]:
# Load the same dataset
dataset = fetch_openml(
    name="diabetes",
    version=1,
    as_frame=True
)
df = dataset.frame
X = dataset.data.copy()
y = dataset.target

X = X.apply(pd.to_numeric, errors="coerce")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [5]:
df.head()


,preg,plas,pres,skin,insu,mass,pedi,age,class
0,6,148,72,35,0,33.6,0.627,50,tested_positive
1,1,85,66,29,0,26.6,0.351,31,tested_negative
2,8,183,64,0,0,23.3,0.672,32,tested_positive
3,1,89,66,23,94,28.1,0.167,21,tested_negative
4,0,137,40,35,168,43.1,2.288,33,tested_positive


# GridSearchCV: Decision Tree

In [7]:
param_grid = {
    "max_depth": [2, 3, 4, 5, 6, None],
    "min_samples_leaf": [1, 5, 10, 20]
}

grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=0),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    return_train_score=True
)

grid_search.fit(X_train, y_train)

best_tree = grid_search.best_estimator_
tree_test_accuracy = accuracy_score(
    y_test, best_tree.predict(X_test)
)

n_combinations = (
    len(param_grid["max_depth"])
    * len(param_grid["min_samples_leaf"])
)

print("Number of combinations:", n_combinations)
print("Total CV fits:", n_combinations * 5)
print("Best parameters:", grid_search.best_params_)
print("Best CV accuracy:", grid_search.best_score_)
print("Test accuracy:", tree_test_accuracy)


Number of combinations: 24
Total CV fits: 120
Best parameters: {'max_depth': 2, 'min_samples_leaf': 10}
Best CV accuracy: 0.7557643609222977
Test accuracy: 0.6948051948051948


# Top 5 configurations by CV rank

In [9]:
results = pd.DataFrame(grid_search.cv_results_)
top_5 = results.sort_values("rank_test_score").head(5)

print(
    top_5[
        [
            "rank_test_score",
            "param_max_depth",
            "param_min_samples_leaf",
            "mean_test_score",
            "std_test_score"
        ]
    ].to_string(index=False)
)

 rank_test_score param_max_depth  param_min_samples_leaf  mean_test_score  std_test_score
               1               2                      20         0.755764        0.028625
               1               2                      10         0.755764        0.028625
               3               3                      10         0.754165        0.038840
               3               3                      20         0.754165        0.038840
               5               2                       1         0.752512        0.031069


# RandomizedSearchCV: Scaled KNN

In [10]:
knn_pipeline = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier()
)

param_distributions = {
    "kneighborsclassifier__n_neighbors": randint(1, 50),
    "kneighborsclassifier__weights": ["uniform", "distance"]
}

random_search = RandomizedSearchCV(
    estimator=knn_pipeline,
    param_distributions=param_distributions,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    random_state=0,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

best_knn = random_search.best_estimator_
knn_test_accuracy = accuracy_score(
    y_test, best_knn.predict(X_test)
)

print("Number of sampled combinations:", 20)
print("Total CV fits:", 20 * 5)
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy:", random_search.best_score_)
print("Test accuracy:", knn_test_accuracy)


Number of sampled combinations: 20
Total CV fits: 100
Best parameters: {'kneighborsclassifier__n_neighbors': 45, 'kneighborsclassifier__weights': 'distance'}
Best CV accuracy: 0.767119818739171
Test accuracy: 0.7142857142857143


# Compare models

In [11]:
print(f"Tuned Decision Tree test accuracy: {tree_test_accuracy:.4f}")
print(f"Tuned KNN test accuracy: {knn_test_accuracy:.4f}")

if tree_test_accuracy > knn_test_accuracy:
    print("Higher test accuracy: Tuned Decision Tree")
elif knn_test_accuracy > tree_test_accuracy:
    print("Higher test accuracy: Tuned KNN")
else:
    print("Both models have the same test accuracy.")

Tuned Decision Tree test accuracy: 0.6948
Tuned KNN test accuracy: 0.7143
Higher test accuracy: Tuned KNN
